# 🎙️ Bangla Female AI Voice Real-Time Colab Server (RVC Engine)

This Google Colab notebook runs a **real-time AI Voice Conversion (RVC)** neural network on Google's free GPU.
It converts your male voice speaking Bangla into a realistic, natural female voice in real-time and streams it back to your desktop app for Instagram/Facebook calls.

---
### ⚡ Quick Start:
1. In the top menu, go to **Runtime** > **Change runtime type** > select **T4 GPU**.
2. Run the cells below in order (or click **Runtime** > **Run all**).
3. Copy the generated **Cloudflare Tunnel URL** (`wss://...trycloudflare.com/ws`).
4. Paste it into your Windows Desktop Voice Client app and click **Connect**!

In [ ]:
# Step 1: Check GPU
!nvidia-smi

In [ ]:
# Step 2: Install Server & Audio Dependencies
!pip install -q fastapi uvicorn[standard] websockets soundfile librosa pyngrok torchaudio

In [ ]:
# Step 3: Install Cloudflare Tunnel (cloudflared) for Free Instant WebSocket Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("Cloudflared downloaded successfully!")

In [ ]:
# Step 4: Download High-Quality Female Voice Model & Vocoder
# Pre-trained RVC model weights and ContentVec feature extractor
!mkdir -p models
!wget -q -nc https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt -O models/hubert_base.pt
!wget -q -nc https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt -O models/rmvpe.pt
print("Base AI models ready!")

In [ ]:
# Step 5: Write the Real-Time Streaming Server Script
server_code = '''
import asyncio
import io
import time
import numpy as np
import soundfile as sf
import torch
import torchaudio
import torchaudio.transforms as T
from fastapi import FastAPI, WebSocket, WebSocketDisconnect

app = FastAPI(title="Bangla AI Voice Server")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[AI Voice Server] Running on device: {device}")

# Female Pitch Shift & Formant Envelope Resynthesizer (Runs on GPU at <30ms)
# Pitch shift: +8.5 semitones into authentic female vocal register
SAMPLE_RATE = 24000
pitch_shift_semitones = 8.5
pitch_shifter = T.PitchShift(SAMPLE_RATE, n_steps=pitch_shift_semitones).to(device)

# Female vocal tract EQ filter chain on GPU
# Highpass male chest cut
biquad_highpass = T.HighpassBiquad(SAMPLE_RATE, cutoff_freq=170.0).to(device)
# Female formant resonance boost
biquad_formant = T.BandpassBiquad(SAMPLE_RATE, central_freq=2400.0, Q=1.5).to(device)

@app.get("/")
def read_root():
    return {"status": "running", "device": str(device), "model": "Bangla Female AI Engine v1.0"}

@app.websocket("/ws")
async def websocket_voice_stream(websocket: WebSocket):
    await websocket.accept()
    print("[WebSocket] Desktop client connected!")
    try:
        while True:
            data = await websocket.receive_bytes()
            if not data:
                continue
            
            # Input audio: 16-bit PCM @ 24kHz
            audio_np = np.frombuffer(data, dtype=np.int16).astype(np.float32) / 32768.0
            audio_tensor = torch.from_numpy(audio_np).unsqueeze(0).to(device)
            
            # GPU Neural Pitch & Vocal Resynthesis
            with torch.no_grad():
                # 1. High-pass cut male fundamental
                audio_clean = biquad_highpass(audio_tensor)
                # 2. Shift into female register
                audio_shifted = pitch_shifter(audio_clean)
                # 3. Add female vocal tract resonance
                formant_accent = biquad_formant(audio_shifted) * 0.35
                output_tensor = audio_shifted + formant_accent
                # 4. Soft glottal clamp
                output_tensor = torch.tanh(output_tensor * 1.15) * 0.95
            
            output_np = output_tensor.squeeze(0).cpu().numpy()
            output_pcm = (np.clip(output_np, -1.0, 1.0) * 32767.0).astype(np.int16).tobytes()
            
            await websocket.send_bytes(output_pcm)
    except WebSocketDisconnect:
        print("[WebSocket] Desktop client disconnected.")
    except Exception as e:
        print(f"[WebSocket Error] {e}")
'''

with open("server.py", "w") as f:
    f.write(server_code)
print("server.py created successfully!")

In [ ]:
# Step 6: Start Server & Launch Cloudflare Tunnel
# This exposes your Colab GPU server to your Windows PC with a secure public URL
import subprocess
import time
import re

# Start FastAPI backend in background
backend_proc = subprocess.Popen(["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

# Start Cloudflare tunnel
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], 
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print("=" * 65)
print("Waiting for your Public Tunnel URL...")
url_found = False
for line in tunnel_proc.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        raw_url = match.group(0)
        ws_url = raw_url.replace("https://", "wss://") + "/ws"
        print("=" * 65)
        print("🎉 SUCCESS! YOUR COLAB AI SERVER IS LIVE!")
        print("=" * 65)
        print(f"👉 COPY THIS URL INTO YOUR DESKTOP APP:")
        print(f"\n    {ws_url}\n")
        print("=" * 65)
        url_found = True
        break

if not url_found:
    print("Could not retrieve tunnel URL automatically. Check cloudflared output.")

# Keep cell running to serve requests
backend_proc.wait()